#Variational Autoencoder (VAE) Implementation on MNIST Dataset

This notebook implements a CNN-based Variational Autoencoder for the MNIST dataset. We'll follow the concepts presented in the slides to build, train, and evaluate a VAE model.
## 1. Setup and Imports

## 1. Setup and Imports

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm

# Set random seed for reproducibility
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


## 2. Data Loading and Preprocessing

In [ ]:
# MNIST dataset
transform = transforms.Compose([
    transforms.ToTensor(),
    # Threshold at 0.5 to create binary images
    lambda x: (x > 0.5).float().clamp(0,1)
])


#train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)

# Training dataset
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)

# Test dataset
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

# Display some example images
plt.figure(figsize=(10, 4))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(train_dataset[i][0].squeeze().numpy(), cmap='gray')
    plt.title(f"Label: {train_dataset[i][1]}")
    plt.axis('off')
plt.tight_layout()
plt.show()


## 3. Baseline: Standard Autoencoder

In [ ]:
# Implement CNN-based autoencoder
class ConvAE(nn.Module):
    def __init__(self, latent_dim=64):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, stride=1, padding=1), # 1 * 28 * 28 -> 16 * 28 * 28
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2), # 16 * 28 * 28 -> 16 * 14 * 14
            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, stride=1, padding=1), # 16 * 14 * 14 -> 32 * 14 * 14
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2), # 32 * 14 * 14 -> 32 * 7 * 7
            nn.Flatten(),
            nn.Linear(in_features=7*7*32, out_features=latent_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(in_features=latent_dim, out_features=7*7*32),
            nn.Unflatten(dim=1, unflattened_size=(32,7,7)),
            nn.Upsample(scale_factor=2, mode='nearest'), # 32 * 7 * 7 -> 32 * 14 * 14
            nn.Conv2d(in_channels=32, out_channels=16, kernel_size=3, stride=1, padding=1), # 32 * 14 * 14 -> 16 * 14 * 14
            nn.ReLU(),
            nn.Upsample(scale_factor=2, mode='nearest'), # 16 * 14 * 14 -> 16 * 28 * 28
            nn.Conv2d(in_channels=16, out_channels=1, kernel_size=3, stride=1, padding=1), # 16 * 28 * 28 -> 1 * 28 * 28
            nn.Sigmoid()
        )

    def forward(self, x):
        # Encode
        latent = self.encoder(x) # z-> latent vector
        # Decode
        reconstructed = self.decoder(latent)
        # Reshape to original image dimensions
        reconstructed = reconstructed.view(-1, 1, 28, 28)
        return reconstructed, latent

    def encode(self, x):
        return self.encoder(x)

    def decode(self, z):
        reconstructed = self.decoder(z)
        return reconstructed.view(-1, 1, 28, 28)

# Binary Cross Entropy Loss
def bce_loss(recon_x, x):
    return nn.BCELoss(reduction='sum')(recon_x, x)


In [ ]:
def train_autoencoder(model, train_loader, test_loader, num_epochs=10,
                      learning_rate=1e-3, device='cuda' if torch.cuda.is_available() else 'cpu'):
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    train_losses = []
    test_losses = []

    for epoch in range(num_epochs):
        # Training
        model.train()
        train_loss = 0
        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
        for batch_idx, (data, _) in enumerate(progress_bar):
            data = data.to(device)
            optimizer.zero_grad()

            # Forward pass
            recon_batch, _ = model(data)
            loss = nn.BCELoss(reduction='sum')(recon_batch, data)


            # Backward pass and optimize
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

            # Update progress bar
            progress_bar.set_postfix(loss=loss.item()/len(data))

        train_loss /= len(train_loader.dataset)
        train_losses.append(train_loss)

        # Validation
        model.eval()
        test_loss = 0
        with torch.no_grad():
            for data, _ in test_loader:
                data = data.to(device)
                recon_batch, _ = model(data)
                loss = nn.BCELoss(reduction='sum')(recon_batch, data)

                test_loss += loss.item()

        test_loss /= len(test_loader.dataset)
        test_losses.append(test_loss)

        print(f'====> Epoch: {epoch+1} Average train loss: {train_loss:.6f}, Test loss: {test_loss:.6f}')

    return train_losses, test_losses



### 3.1 Training the Autoencoder

In [ ]:
# Setting hyperparameters
latent_dim = 64
num_epochs = 6  # Can be increased for better results

# Initialize models
ae_model = ConvAE(latent_dim=latent_dim)


# Train with loss
print("Training autoencoder with  loss...")
train_losses, test_losses = train_autoencoder(ae_model, train_loader, test_loader, num_epochs=num_epochs, device=device)

### 3.2 Visualize Reconstruction

In [ ]:
# Function to visualize reconstructions
def visualize_reconstructions(model, data_loader, title, num_images=10,
                             device='cuda' if torch.cuda.is_available() else 'cpu'):
    model.eval()
    with torch.no_grad():
        for batch_idx, (data, _) in enumerate(data_loader):
            original_data = data[:num_images].to(device)
            reconstructed_data, _ = model(original_data)

            plt.figure(figsize=(20, 4))

            # Original images
            for i in range(num_images):
                plt.subplot(2, num_images, i + 1)
                plt.imshow(original_data[i].cpu().numpy().reshape(28, 28), cmap='gray')
                plt.title('Original')
                plt.axis('off')

            # Reconstructed images
            for i in range(num_images):
                plt.subplot(2, num_images, num_images + i + 1)
                plt.imshow(reconstructed_data[i].cpu().numpy().reshape(28, 28), cmap='gray')
                plt.title('Reconstructed')
                plt.axis('off')

            plt.suptitle(title)
            plt.tight_layout()
            plt.show()
            break  # Only plot one batch

# Visualize reconstructions for each model
visualize_reconstructions(ae_model, test_loader, "MSE Autoencoder Reconstructions")

### 3.3 Sampling (Failure Case)

In [ ]:
# Sample from latent space
def sample_latent_space(model, num_samples=10, latent_dim=20,
                        device='cuda' if torch.cuda.is_available() else 'cpu'):
    model.eval()
    with torch.no_grad():
        # Generate random points in latent space
        z = torch.randint(5, size=(num_samples, latent_dim)).float().to(device)
        # Decode the random points
        samples = model.decode(z)

        plt.figure(figsize=(15, 3))
        for i in range(num_samples):
            plt.subplot(1, num_samples, i + 1)
            plt.imshow(samples[i].cpu().numpy().reshape(28, 28), cmap='gray')
            plt.axis('off')
        plt.suptitle("Random Samples from Latent Space")
        plt.tight_layout()
        plt.show()

# Sample from the latent space of the combined loss model
sample_latent_space(ae_model, num_samples=10, latent_dim=latent_dim, device=device)


## 4. VAE Loss Components

### 4.1 Reconstruction Loss Definition

In [ ]:
def reconstruction_loss(x, x_recon):
    """
    Binary Cross-Entropy loss for Bernoulli likelihood model.

    For binary data (MNIST):
    -log p(x|z) = -∑[x_i * log(θ_i(z)) + (1-x_i) * log(1-θ_i(z))]

    where θ_i(z) is the probability of pixel i being 1, decoded from latent code z.
    This matches the Bernoulli likelihood description from slide 133.
    """
    # BCELoss expects inputs in range [0, 1]
    # Each pixel value in x_recon is interpreted as the probability of that pixel being 1
    BCE = F.binary_cross_entropy(x_recon, x, reduction='sum')
    return BCE


### 4.2 KL Divergence Loss Definition

In [ ]:
def kl_divergence(mu, logvar):
    """
    KL divergence between q(z|x) = N(μ(x), σ²(x)) and p(z) = N(0, I).

    As shown in slides:
    D_KL(q(z|x) || p(z)) = 1/2 ∑(μ² + σ² - log(σ²) - 1)

    For the standard normal prior p(z) = N(0, I), this simplifies to the formula below.
    """
    # -0.5 * sum(1 + log(σ²) - μ² - σ²)
    KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return KLD


## 5. KL Annealing Implementation

In [ ]:
def get_beta(step, warmup_steps, max_beta=1.0):
    """
    Linear KL annealing schedule (as shown in slide 142).

    Args:
        step: Current training step
        warmup_steps: Number of warmup steps
        max_beta: Maximum β value (default: 1.0)

    Returns:
        β value for the current step
    """
    if step > warmup_steps:
        return max_beta
    else:
        return max_beta * (step / warmup_steps)


## 6. CNN-Based VAE Model Implementation

In [ ]:
class VAE(nn.Module):
    def __init__(self, latent_dim=4):
        super(VAE, self).__init__()
        self.latent_dim = latent_dim

        # Encoder
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=8, kernel_size=3, stride=1, padding=1),  # 28x28 -> 28x28
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),  # 28x28 -> 14x14
            nn.Conv2d(in_channels=8, out_channels=16, kernel_size=3, stride=1, padding=1),  # 14x14 -> 14x14
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),  # 14x14 -> 7x7
            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, stride=1, padding=1),  # 7x7 -> 7x7
            nn.ReLU(),
            nn.Flatten()
        )

        # Calculate the size of the encoder output
        self.encoder_output_size = 32 * 7 * 7

        # Mean and log-variance layers
        self.fc_mu = nn.Linear(in_features=self.encoder_output_size, out_features=latent_dim)
        self.fc_logvar = nn.Linear(in_features=self.encoder_output_size, out_features=latent_dim)

        # Decoder initial fully connected layer
        self.decoder_input = nn.Linear(in_features=latent_dim, out_features=7 * 7 * 32)

        # Decoder
        self.decoder = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='nearest'),  # 7x7 -> 14x14
            nn.Conv2d(in_channels=32, out_channels=16, kernel_size=3, stride=1, padding=1),  # 14x14 -> 14x14
            nn.ReLU(),
            nn.Upsample(scale_factor=2, mode='nearest'),  # 14x14 -> 28x28
            nn.Conv2d(in_channels=16, out_channels=8, kernel_size=3, stride=1, padding=1),  # 28x28 -> 28x28
            nn.ReLU(),
            nn.Conv2d(in_channels=8, out_channels=1, kernel_size=3, stride=1, padding=1),  # 28x28 -> 28x28
            nn.Sigmoid()  # Output pixels in [0,1] range for BCE loss
        )

    def encode(self, x):
        h = self.encoder(x)
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        logvar = torch.clamp(logvar, min=-10, max=10)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        """
        Reparameterization trick as described in slides 109-110.
        Instead of sampling directly from q(z|x), we sample from
        epsilon ~ N(0,I) and compute z = mu + sigma * epsilon
        """
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        h = self.decoder_input(z)
        h = h.view(-1, 32, 7, 7)  # Reshape for ConvTranspose layers
        return self.decoder(h)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        x_recon = self.decode(z)
        return x_recon, mu, logvar

# Initialize model
model = VAE().to(device)
print(model)


## 7. Training Loop with KL Annealing

In [ ]:
def train(model, train_loader, optimizer, epochs=30, warmup_steps=10000, max_beta=1.0):
    model.train()
    train_losses = []
    kl_losses = []
    recon_losses = []
    step = 0

    for epoch in range(epochs):
        epoch_loss = 0
        epoch_recon_loss = 0
        epoch_kl_loss = 0

        progress_bar = tqdm(enumerate(train_loader), total=len(train_loader), desc=f"Epoch {epoch+1}/{epochs}")

        for i, (x, _) in progress_bar:

            x = x.to(device)
            optimizer.zero_grad()

            # Forward pass
            x_recon, mu, logvar = model(x)

            # Calculate losses
            recon_loss = reconstruction_loss(x, x_recon)
            kl_loss = kl_divergence(mu, logvar)

            # Apply KL annealing
            beta = get_beta(step, warmup_steps, max_beta)

            # Total loss (following the ELBO objective from slide 133)
            loss = recon_loss + beta * kl_loss

            # Backward pass and optimize
            loss.backward()
            optimizer.step()

            # Update metrics
            epoch_loss += loss.item()
            epoch_recon_loss += recon_loss.item()
            epoch_kl_loss += kl_loss.item()

            # Update progress bar
            progress_bar.set_postfix({
                'loss': loss.item() / len(x),
                'recon_loss': recon_loss.item() / len(x),
                'kl_loss': kl_loss.item() / len(x),
                'beta': beta
            })

            step += 1

        # Compute average losses
        avg_loss = epoch_loss / len(train_loader.dataset)
        avg_recon_loss = epoch_recon_loss / len(train_loader.dataset)
        avg_kl_loss = epoch_kl_loss / len(train_loader.dataset)

        train_losses.append(avg_loss)
        recon_losses.append(avg_recon_loss)
        kl_losses.append(avg_kl_loss)

        print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}, Recon Loss: {avg_recon_loss:.4f}, KL Loss: {avg_kl_loss:.4f}, Beta: {beta:.4f}")

    return train_losses, recon_losses, kl_losses

## 8. Visualizing the 2D Latent Space

In [ ]:
# Create VAE with 2D latent space
model_2d = VAE(latent_dim=2).to(device)
optimizer_2d = optim.Adam(model_2d.parameters(), lr=1e-3)

# Train the 2D VAE (using fewer epochs for speed in demonstration)
print("Training 2D VAE...")
train_losses_2d, recon_losses_2d, kl_losses_2d = train(model_2d, train_loader, optimizer_2d, epochs=6, warmup_steps=2000)

import numpy as np
import matplotlib.pyplot as plt

# 1. Plot Grid Search Generation
def plot_2d_latent_grid(model, n=15, digit_size=28):
    figure = np.zeros((digit_size * n, digit_size * n))
    grid_x = np.linspace(-3, 3, n)
    grid_y = np.linspace(-3, 3, n)[::-1]

    model.eval()
    with torch.no_grad():
        for i, yi in enumerate(grid_y):
            for j, xi in enumerate(grid_x):
                z_sample = torch.tensor([[xi, yi]]).float().to(device)
                x_decoded = model.decode(z_sample)
                digit = x_decoded[0].cpu().squeeze().numpy()
                figure[i * digit_size: (i + 1) * digit_size,
                       j * digit_size: (j + 1) * digit_size] = digit

    plt.figure(figsize=(10, 10))
    plt.imshow(figure, cmap='gray')
    plt.axis('off')
    plt.title("Grid Search of 2D Latent Space")
    plt.show()

print("Plotting latent grid...")
plot_2d_latent_grid(model_2d)

# 2. Plot Scatter Plot of Latent Space
def plot_2d_latent_space(model, data_loader):
    model.eval()
    all_mu = []
    all_labels = []

    with torch.no_grad():
        for x, y in data_loader:
            x = x.to(device)
            mu, _ = model.encode(x)
            all_mu.append(mu.cpu().numpy())
            all_labels.append(y.numpy())

    all_mu = np.concatenate(all_mu, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)

    plt.figure(figsize=(10, 8))
    scatter = plt.scatter(all_mu[:, 0], all_mu[:, 1], c=all_labels, cmap='tab10', alpha=0.5, s=2)
    plt.colorbar(scatter, ticks=range(10))
    plt.xlabel('z[0]')
    plt.ylabel('z[1]')
    plt.title('2D Latent Space Projection')
    plt.show()

print("Plotting latent space projection...")
plot_2d_latent_space(model_2d, test_loader)

### Regular VAE Training

In [ ]:
# Initialize optimizer (Adam as recommended in the slides)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# Train model
print("Starting training...")
train_losses, recon_losses, kl_losses = train(model, train_loader, optimizer, epochs=16, warmup_steps=2000)
print("Training complete!")

## 9. Plot Training Metrics

In [ ]:
# Plot training losses
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.plot(train_losses)
plt.title('Total Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')

plt.subplot(1, 3, 2)
plt.plot(recon_losses)
plt.title('Reconstruction Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')

plt.subplot(1, 3, 3)
plt.plot(kl_losses)
plt.title('KL Divergence Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')

plt.tight_layout()
plt.show()

## 10. Evaluate and Visualize Reconstructions

In [ ]:
def visualize_reconstructions(model, test_loader, n=10):
    model.eval()

    # Get batch of test images
    dataiter = iter(test_loader)
    images, labels = next(dataiter)
    images = images[:n].to(device)

    # Reconstruct images
    with torch.no_grad():
        reconstructions, _, _ = model(images)

    # Plot original vs reconstructed images
    plt.figure(figsize=(20, 4))
    for i in range(n):
        # Original images
        ax = plt.subplot(2, n, i + 1)
        plt.imshow(images[i].cpu().squeeze().numpy(), cmap='gray')
        plt.title(f"Original: {labels[i]}")
        plt.axis('off')

        # Reconstructed images
        ax = plt.subplot(2, n, n + i + 1)
        plt.imshow(reconstructions[i].cpu().squeeze().numpy(), cmap='gray')
        plt.title(f"Reconstructed")
        plt.axis('off')

    plt.tight_layout()
    plt.show()

# Visualize reconstructions
visualize_reconstructions(model, test_loader)


## 11. Generate Samples from the Latent Space

In [ ]:
def generate_samples(model, num_samples=10):
    model.eval()

    with torch.no_grad():
        # Sample from the latent space (prior p(z) = N(0, I))
        z = torch.randn(num_samples, model.latent_dim).to(device)

        # Decode samples
        samples = model.decode(z)

    # Plot generated samples
    plt.figure(figsize=(15, 3))
    for i in range(num_samples):
        plt.subplot(1, num_samples, i + 1)
        plt.imshow(samples[i].cpu().squeeze().numpy(), cmap='gray')
        plt.title(f"Sample {i+1}")
        plt.axis('off')

    plt.tight_layout()
    plt.show()

# Generate samples
generate_samples(model)


## 12. Latent Space Interpolation

In [ ]:
def interpolate_latent_space(model, test_loader, steps=10):
    model.eval()

    # Get two test images
    dataiter = iter(test_loader)
    images, labels = next(dataiter)
    image1 = images[50:51].to(device)
    image2 = images[3:4].to(device)

    # Encode images to get latent representations
    with torch.no_grad():
        mu1, logvar1 = model.encode(image1)
        mu2, logvar2 = model.encode(image2)

        # Use means as the representations
        z1 =  mu1
        z2 =  mu2

    # Interpolate between the latent representations
    interpolations = []
    for alpha in np.linspace(0, 1, steps):
        z_interp = z1 * (1 - alpha) + z2 * alpha
        # Decode interpolated latent vector
        with torch.no_grad():
            interp_image = model.decode(z_interp)
            interpolations.append(interp_image.cpu().squeeze().numpy())

    # Plot original images and interpolations
    plt.figure(figsize=(15, 3))

    # Original image 1
    plt.subplot(1, steps + 2, 1)
    plt.imshow(image1.cpu().squeeze().numpy(), cmap='gray')
    plt.title(f"Original: {labels[0]}")
    plt.axis('off')

    # Interpolations
    for i in range(steps):
        plt.subplot(1, steps + 2, i + 2)
        plt.imshow(interpolations[i], cmap='gray')
        plt.title(f"α={i/(steps-1):.1f}")
        plt.axis('off')

    # Original image 2
    plt.subplot(1, steps + 2, steps + 2)
    plt.imshow(image2.cpu().squeeze().numpy(), cmap='gray')
    plt.title(f"Original: {labels[1]}")
    plt.axis('off')

    plt.tight_layout()
    plt.show()

# Interpolate in latent space
interpolate_latent_space(model, test_loader)


## 13. Student Tasks (TODOs)

### 13.1 β-VAE: Exploring the Effect of the Beta Hyperparameter

In [ ]:
# TODO: Modify your training loop to accept a beta parameter
# Example: loss = recon_loss + beta * kl_loss
beta = 4.0  # Try different values
# ... rest of your training loop here


### 13.2 Latent Traversals

In [ ]:
# TODO: Implement latent traversal visualization
# For each dimension, vary z_i in [-3, 3] while fixing others


### 13.3 VAE on CIFAR-100

In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import random

# Define transformation: ToTensor() converts images to [0,1] float tensors
transform = transforms.Compose([
    transforms.ToTensor()
])

# Load Fashion-MNIST training and test datasets
train_dataset = datasets.CIFAR100(
    root='data',
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.CIFAR100(
    root='data',
    train=False,
    download=True,
    transform=transform
)

# Create DataLoaders
batch_size = 128

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

# Display some example images
plt.figure(figsize=(10, 4))
for idx, i in enumerate(random.sample(range(128),10)):
    plt.subplot(2, 5, idx + 1)
    plt.imshow(train_dataset[i][0].permute(1, 2, 0).numpy())
    plt.title(f"Label: {train_dataset[i][1]}")
    plt.axis('off')
plt.tight_layout()
plt.show()

## 📝 Student Information

Please fill in your details below before submitting.

- **Student Name:** *Type your name here*
- **Roll Number:** *Type your roll number here*
- **Date of Completion:** *Type date here*

## 📤 Submission Instructions

Please submit your completed `.ipynb` notebook file to the Google Form linked below:

🔗 [Submit Here](https://forms.gle/aymKUh52hHsfS5CM7)

⚠️ **Deadline:** April 08, 2026 at 11:59 PM GMT+5:30